# Phase 3 — Finalize table/stage registration

Verify `CURATED.SENSORS` / `CURATED.SPLITS`, written by Phase 2
(`jobs/phase2_extract_organize.py`), against Phase 3's acceptance criteria.
The tables themselves were already created by Phase 2's ML Job via
`write_pandas(auto_create_table=True)`. Here we **only confirm they were built
correctly** (no re-extraction).

**What this notebook does, cell by cell:**

1. Point the session at `CAR_MULTIMODAL_AI_DB.CURATED` (cell 2).
2. Check the train/val/test split ratio is roughly 70/15/15 (cell 4).
3. Check the lead-vehicle distance distribution matches the known baseline (cell 6).
4. Check the fraction of frames with a valid distance label is close to 77.8% (cell 8).
5. Check the `SEGMENTS` ledger for any segments Phase 1/2 marked as failed/corrupt (cell 10).

If all four checks look right, the data foundation is trustworthy and it is safe to
proceed to Phase 4 (EDA) and Phase 5/6 (feature extraction / model development).
**No data is modified by this notebook** — every cell is read-only `SELECT`.


### Cell: set the session's current database/schema

A fresh Notebook session does not automatically inherit the database/schema that
Phase 2's ML Job used. `get_active_session()` grabs the session Snowsight already
opened for this Notebook; the two `USE ...` statements point it at
`CAR_MULTIMODAL_AI_DB.CURATED`, so every unqualified table name in the rest of this
notebook (`SPLITS`, `SENSORS`) resolves correctly without needing a fully-qualified
`CAR_MULTIMODAL_AI_DB.CURATED.SPLITS` on every line. Nothing is printed by this cell;
if it errors, the database/schema created by `scripts/setup.py` doesn't exist yet.


In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
session.sql("USE DATABASE CAR_MULTIMODAL_AI_DB").collect()
session.sql("USE SCHEMA CURATED").collect()

## Acceptance criterion 1: the split ratio is roughly 70/15/15

### Cell: count rows per split value

Runs `SELECT split, COUNT(*) FROM SPLITS GROUP BY split`, i.e. one row per distinct
value of `SPLITS.SPLIT` (`train` / `val` / `test`), then adds a `pct` column showing
each group's share of the total row count.

**What a healthy result looks like:** three rows, with `pct` close to
`70 / 15 / 15`. This is a *segment-level* split (whole driving segments are assigned
to a split, not individual frames), and deliberately so: frames inside the same
segment are correlated, so a per-frame random split would leak
information from train into test and make the later ablation results look better
than they really are. A skewed ratio (e.g. 95/3/2) would mean the segment-level
splitting logic in Phase 2 has a bug.


In [ ]:
df_split = session.sql(
    "SELECT split, COUNT(*) AS n FROM SPLITS GROUP BY split ORDER BY split"
).to_pandas()
df_split["pct"] = 100 * df_split["N"] / df_split["N"].sum()
df_split

## Acceptance criterion 2: the distance distribution matches the known baseline
(median 47.8m, p10=9.0m, range 3.0–149.8m)

### Cell: descriptive statistics of the DISTANCE column

Pulls every non-null `DISTANCE` value (the radar-derived lead-vehicle distance
label, in meters) into a pandas Series and calls `.describe()` with the 10th/50th
(median)/90th percentiles added. The output is a small table with rows
`count, mean, std, min, 10%, 50%, 90%, max`.

**What to compare:** `50%` (median) should be near **47.8 m**, `10%` near **9.0 m**,
and `min`/`max` roughly **3.0 m** to **149.8 m**. These numbers are the known
baseline for this dataset, measured in advance on Chunk_1 — matching
them closely (not exactly, since the underlying HEVC decode/label-generation timing
has some non-determinism) confirms Phase 2's label-generation logic (nearest
in-lane radar track, matched to camera frame timestamps) was ported to Snowflake
without a scaling or unit-conversion mistake.


In [ ]:
df_dist = session.sql(
    "SELECT DISTANCE FROM SENSORS WHERE DISTANCE IS NOT NULL"
).to_pandas()
df_dist["DISTANCE"].describe(percentiles=[0.1, 0.5, 0.9])

## Acceptance criterion 3: the valid-label rate does not deviate far from 77.8%
(target: within ±5pt in the narrative baseline; the code below uses a wider ±10pt
tolerance so this cell isn't flaky run-to-run — see the printed number for the
actual value)

### Cell: fraction of frames with a usable label

Not every camera frame has a lead-vehicle radar match within the ±0.1s window used
by Phase 2's label generator (e.g. no car ahead, or no in-lane radar track at that
exact instant) — those frames have `DISTANCE = NULL` and are excluded from
training. This cell computes:

```
valid_rate = 100 * (rows where DISTANCE IS NOT NULL) / (total rows)
```

prints it next to the 77.8% baseline, then `assert`s it is within 10 percentage
points of that baseline. **If this assertion fails**, the label-generation logic
in `jobs/phase2_extract_organize.py` likely has a bug
(wrong radar time window, wrong ego-lane threshold, etc.) — re-check Phase 2
before trusting anything downstream.


In [ ]:
n_total = session.sql("SELECT COUNT(*) AS N FROM SENSORS").to_pandas().iloc[0, 0]
n_valid = session.sql("SELECT COUNT(*) AS N FROM SENSORS WHERE DISTANCE IS NOT NULL").to_pandas().iloc[0, 0]
valid_rate = 100 * n_valid / n_total
print(f"valid label rate: {valid_rate:.1f}% (baseline: 77.8%)")
assert abs(valid_rate - 77.8) <= 10, "Valid label rate deviates significantly from expectations. Check Phase 2's label generation logic."

## Check the SEGMENTS ledger status (any corrupt segments?)

### Cell: per-status segment counts

`RAW.SEGMENTS` is the ledger Phase 1 (ingestion) and Phase 2 (extraction) both
write to, one row per driving segment, with a `STATUS` column
(`staged` / `corrupt` / etc.). This cell just runs
`SELECT STATUS, COUNT(*) FROM RAW.SEGMENTS GROUP BY STATUS` and displays the
result as a small table.

**What a healthy result looks like:** a single row, `STATUS='staged'`, with a count
matching the number of segments in the ingested chunk(s) (188 for the default
`Chunk_1`). Any `corrupt` rows mean Phase 1/2 hit a download or decode failure on
those specific segments — those segments are silently excluded from `SENSORS`, so
a nonzero `corrupt` count here is informational (it explains why the frame count in
criterion 3 might be slightly lower than expected), not necessarily an error to fix.


In [ ]:
session.sql(
    "SELECT STATUS, COUNT(*) AS N FROM RAW.SEGMENTS GROUP BY STATUS"
).to_pandas()

Once all checks pass, Phase 3 is complete. Next is
`notebooks/phase4_eda.ipynb` (data visualization/analysis).
